# PySpark DataFrame Data Engineering Project — Automotive Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `vehicles.csv`, `service_records.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("vehicles.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("service_records.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select vehicle_id and city from the first DataFrame.

In [ ]:
df1.select('vehicle_id','city').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where service_cost is greater than its average.

In [ ]:
avg_value = df2.select(avg('service_cost').alias('avg')).first()['avg']
df2.filter(col('service_cost') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on service_cost using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('service_cost') >= avg('service_cost').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in city and service_type with 'Unknown'.

In [ ]:
df1.fillna({'city': 'Unknown'}).show(5)
df2.fillna({'service_type': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using vehicle_id and vehicle_id.

In [ ]:
joined = df1.join(df2, df1['vehicle_id'] == df2['vehicle_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['vehicle_id'] == df2['vehicle_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average service_cost by service_type.

In [ ]:
df2.groupBy('service_type').agg(count('*').alias('records'), avg('service_cost').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total service_cost by service_type.

In [ ]:
df2.groupBy('service_type').agg(sum('service_cost').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum service_cost in each service_type.

In [ ]:
df2.groupBy('service_type').agg(max('service_cost').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum service_cost in each service_type.

In [ ]:
df2.groupBy('service_type').agg(min('service_cost').alias('min_value')).show()

## Q14. Having Condition

Show service_type groups whose total service_cost is greater than the overall average service_cost.

In [ ]:
overall = df2.select(avg('service_cost').alias('v')).first()['v']
df2.groupBy('service_type').agg(sum('service_cost').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by service_cost.

In [ ]:
df2.orderBy(desc('service_cost')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by service_cost.

In [ ]:
df2.orderBy(asc('service_cost')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by service_cost within each service_type.

In [ ]:
w = Window.partitionBy('service_type').orderBy(desc('service_cost'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of service_cost within each service_type.

In [ ]:
w = Window.partitionBy('service_type').orderBy('service_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('service_cost').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's service_cost with the previous record in its service_type.

In [ ]:
w = Window.partitionBy('service_type').orderBy('service_date')
df2.withColumn('previous_value', lag('service_cost').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each service_type's contribution to total service_cost.

In [ ]:
tot = df2.select(sum('service_cost').alias('t')).first()['t']
df2.groupBy('service_type').agg(sum('service_cost').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from service_date.

In [ ]:
df2.withColumn('year', year('service_date')).withColumn('month', month('service_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average service_cost by month.

In [ ]:
df2.withColumn('month', date_format('service_date','yyyy-MM')).groupBy('month').agg(sum('service_cost').alias('total'), avg('service_cost').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with service_type as rows and year of service_date as columns.

In [ ]:
df2.withColumn('year', year('service_date')).groupBy('service_type').pivot('year').agg(sum('service_cost')).show()

## Q24. Conditional KPI

Count records where service_cost is above the category average.

In [ ]:
w=Window.partitionBy('service_type')
df2.withColumn('cat_avg',avg('service_cost').over(w)).filter(col('service_cost')>col('cat_avg')).groupBy('service_type').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total service_cost by city.

In [ ]:
df1.join(df2, df1['vehicle_id']==df2['vehicle_id']).groupBy(df1['city']).agg(sum(df2['service_cost']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank city by total service_cost.

In [ ]:
r=df1.join(df2,df1['vehicle_id']==df2['vehicle_id']).groupBy(df1['city']).agg(sum(df2['service_cost']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for service_cost.

In [ ]:
df2.withColumn('segment',when(col('service_cost')<1000,'Low').when(col('service_cost')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative service_cost values.

In [ ]:
df2.filter(col('service_cost')<0).count()

## Q29. Final Business Report

Create a final report by service_type with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('service_type').agg(count('*').alias('records'),sum('service_cost').alias('total'),avg('service_cost').alias('average'),min('service_cost').alias('minimum'),max('service_cost').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project